Phase 6 v3 — Two-Tower with **in-batch sampled softmax** instead of BCE on 1:4 explicit negatives. 06 (v2) hit Recall@200 = 0.224, far below the 0.90 target — diagnosis was that BCE on 4 random negatives doesn't push the embedding space hard enough for full 9K candidate ranking. DSSM standard practice is in-batch softmax: every other positive in the batch acts as a negative for the current user, giving B-1 negatives per query at no extra compute cost.

Architecture is identical to 06 (item tower includes ColBERT-light text emb). Only the loss changes:
- v2: `BCE(sigmoid(u·i * T), label)` over (positive + 4 negatives) per row
- v3: `CrossEntropy(u_batch @ i_batch.T * T, arange(B))` over positives-only batch, every other in-batch item is a negative

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import platform

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"arch: {platform.machine()}  device: {DEVICE}")

Inline IDEncoder + Dataset (positives-only) + UserTower / ItemTower.

In [ ]:
class IDEncoder:
    def __init__(self, ids, oov_token="<UNK>"):
        oov_markers = {"<NEW_USER>", "<NEW_BUSINESS>", "<UNK>", oov_token}
        unique_real_ids = sorted({i for i in ids if i not in oov_markers})
        self.id_to_idx = {oov_token: 0}
        for idx, _id in enumerate(unique_real_ids, start=1):
            self.id_to_idx[_id] = idx
        for marker in oov_markers:
            self.id_to_idx.setdefault(marker, 0)
        self._size = 1 + len(unique_real_ids)
    def __len__(self): return self._size
    def encode(self, _id): return self.id_to_idx.get(_id, 0)
    def encode_array(self, ids):
        return ids.map(self.id_to_idx).fillna(0).astype(np.int64).values


class PositivesOnlyDataset(Dataset):
    """In-batch softmax requires positives-only batches (in-batch negatives free)."""
    def __init__(self, df_pos, user_encoder, item_encoder, user_features, item_features):
        self.user_idx = torch.from_numpy(user_encoder.encode_array(df_pos["user_id"]))
        self.item_idx = torch.from_numpy(item_encoder.encode_array(df_pos["business_id"]))
        n_users = len(user_encoder); n_items = len(item_encoder)
        self.user_num = np.zeros((n_users, 6), dtype=np.float32)
        self.user_cuisine = np.zeros((n_users, 50), dtype=np.float32)
        for _, row in user_features.iterrows():
            uidx = user_encoder.encode(row["user_id"])
            self.user_num[uidx] = [row["avg_rating_given"], row["review_count_log"], row["days_active"],
                                    row["elite_flag"], row["mean_distance_traveled"], row["price_tolerance_avg"]]
            emb = row["fav_cuisine_emb"]
            if isinstance(emb, list) and len(emb) == 50:
                self.user_cuisine[uidx] = emb
        self.item_num = np.zeros((n_items, 7), dtype=np.float32)
        self.item_cat = np.zeros((n_items, 50), dtype=np.float32)
        self.item_text = np.zeros((n_items, 32), dtype=np.float32)
        for _, row in item_features.iterrows():
            iidx = item_encoder.encode(row["business_id"])
            self.item_num[iidx] = [row["avg_rating"], row["review_count_log"], row["price_level"],
                                    row["is_open"], row["has_outdoor_seating"], row["photo_count_log"], row["city_id"]]
            cat = row["categories_multi_hot"]
            if isinstance(cat, list) and len(cat) == 50:
                self.item_cat[iidx] = cat
            text = row.get("item_text_emb_pca32")
            if isinstance(text, (list, np.ndarray)) and len(text) == 32:
                self.item_text[iidx] = np.asarray(text, dtype=np.float32)
        self.user_num_t = torch.from_numpy(self.user_num)
        self.user_cuisine_t = torch.from_numpy(self.user_cuisine)
        self.item_num_t = torch.from_numpy(self.item_num)
        self.item_cat_t = torch.from_numpy(self.item_cat)
        self.item_text_t = torch.from_numpy(self.item_text)
    def __len__(self): return len(self.user_idx)
    def __getitem__(self, idx):
        u = self.user_idx[idx]; i = self.item_idx[idx]
        return {
            "user_idx": u, "item_idx": i,
            "user_num": self.user_num_t[u], "user_cuisine": self.user_cuisine_t[u],
            "item_num": self.item_num_t[i], "item_cat": self.item_cat_t[i],
            "item_text": self.item_text_t[i],
        }


class UserTower(nn.Module):
    def __init__(self, n_users, emb_dim=8, num_dim=6, cuisine_dim=50, hidden=(128, 64), out_dim=32, dropout=0.1):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        layers = []; prev = emb_dim + num_dim + cuisine_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]; prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.mlp = nn.Sequential(*layers)
    def forward(self, user_idx, user_num, user_cuisine):
        u = self.user_emb(user_idx)
        x = torch.cat([u, user_num, user_cuisine], dim=-1)
        return F.normalize(self.mlp(x), dim=-1)


class ItemTower(nn.Module):
    def __init__(self, n_items, emb_dim=8, num_dim=7, cat_dim=50, text_dim=32, hidden=(128, 64), out_dim=32, dropout=0.1):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        layers = []; prev = emb_dim + num_dim + cat_dim + text_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]; prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.mlp = nn.Sequential(*layers)
    def forward(self, item_idx, item_num, item_cat, item_text):
        i = self.item_emb(item_idx)
        x = torch.cat([i, item_num, item_cat, item_text], dim=-1)
        return F.normalize(self.mlp(x), dim=-1)


class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, out_dim=32, temperature=10.0):
        super().__init__()
        self.user_tower = UserTower(n_users, out_dim=out_dim)
        self.item_tower = ItemTower(n_items, out_dim=out_dim)
        self.temperature = temperature

    def encode_user(self, user_idx, user_num, user_cuisine):
        return self.user_tower(user_idx, user_num, user_cuisine)

    def encode_item(self, item_idx, item_num, item_cat, item_text):
        return self.item_tower(item_idx, item_num, item_cat, item_text)

Build positives-only train table + val pool + tower batches. **Important**: in-batch softmax filters out the 1:4 negatives that Phase 3 generated — those are needed for DeepFM, not Two-Tower v3.

In [ ]:
user_features = pd.read_parquet(FEATURES_DIR / "user_features.parquet")
item_features = pd.read_parquet(FEATURES_DIR / "item_features.parquet")
val_df = pd.read_parquet(CLEANED_DIR / "val_reviews.parquet")
train_df_full = pd.read_parquet(FEATURES_DIR / "train_with_negatives.parquet")
train_df_pos = train_df_full[train_df_full["label"] == 1].reset_index(drop=True)
print(f"positives only train: {len(train_df_pos)} (was {len(train_df_full)} with negatives)")

user_encoder = IDEncoder(user_features["user_id"].tolist(), oov_token="<NEW_USER>")
item_encoder = IDEncoder(item_features["business_id"].tolist(), oov_token="<NEW_BUSINESS>")
n_users, n_items = len(user_encoder), len(item_encoder)

t0 = time.time()
train_ds = PositivesOnlyDataset(train_df_pos, user_encoder, item_encoder, user_features, item_features)
print(f"train ds built in {time.time()-t0:.1f}s")
train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True, num_workers=0, pin_memory=False, drop_last=True)
# drop_last=True so every batch has size B; in-batch softmax needs square matrices

**In-batch softmax loss**:

```
u, i = encode_user(B), encode_item(B)   # both (B, 32) normalized
scores = u @ i.T * temperature          # (B, B) — diagonal = true pairs
loss = CrossEntropy(scores, arange(B))  # softmax each row, target = its own diagonal
```
Each row's softmax distributes probability across all B items in batch; the loss pushes the diagonal up and everything else down. This gives B-1 = 4095 negatives per query for free, vs 4 with explicit negatives.

In [ ]:
CONFIG = {"out_dim": 32, "temperature": 20.0, "lr": 1e-3, "l2": 1e-5, "epochs": 10}
print(f"config: {CONFIG}")

model = TwoTower(n_users, n_items, out_dim=CONFIG["out_dim"], temperature=CONFIG["temperature"]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["l2"])
ce = nn.CrossEntropyLoss()

history = {"epoch": [], "train_loss": []}
for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    loss_sum, n_batch = 0.0, 0
    t0 = time.time()
    for batch in train_loader:
        B = batch["user_idx"].size(0)
        u_idx = batch["user_idx"].to(DEVICE)
        i_idx = batch["item_idx"].to(DEVICE)
        un = batch["user_num"].to(DEVICE)
        uc = batch["user_cuisine"].to(DEVICE)
        inum = batch["item_num"].to(DEVICE)
        ic = batch["item_cat"].to(DEVICE)
        it = batch["item_text"].to(DEVICE)

        u_emb = model.encode_user(u_idx, un, uc)         # (B, 32)
        i_emb = model.encode_item(i_idx, inum, ic, it)   # (B, 32)
        scores = (u_emb @ i_emb.T) * model.temperature   # (B, B)
        labels = torch.arange(B, device=DEVICE)
        loss = ce(scores, labels)

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_sum += loss.item(); n_batch += 1
    train_loss = loss_sum / n_batch
    elapsed = time.time() - t0
    print(f"epoch {epoch:02d} | {elapsed:.0f}s | loss={train_loss:.4f}")
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)

torch.save(model.state_dict(), MODELS_DIR / "two_tower_v3_inbatch.pt")

Pre-compute item index, eval Recall@K against the full 9K pool. Same eval protocol as v2 so the comparison is apples-to-apples.

In [ ]:
@torch.no_grad()
def precompute_item_index(model, n_items, dataset, device, batch_size=4096):
    model.eval()
    all_idx = np.arange(n_items, dtype=np.int64)
    out = np.zeros((n_items, 32), dtype=np.float32)
    for start in range(0, n_items, batch_size):
        end = min(start + batch_size, n_items)
        idx_t = torch.from_numpy(all_idx[start:end]).to(device)
        emb = model.encode_item(
            idx_t,
            dataset.item_num_t[start:end].to(device),
            dataset.item_cat_t[start:end].to(device),
            dataset.item_text_t[start:end].to(device),
        ).cpu().numpy()
        out[start:end] = emb
    return out

t0 = time.time()
item_index = precompute_item_index(model, n_items, train_ds, DEVICE)
print(f"item index: {item_index.shape} in {time.time()-t0:.2f}s")
np.save(MODELS_DIR / "two_tower_v3_item_index.npy", item_index)


@torch.no_grad()
def eval_recall_at_k(model, val_pos, user_encoder, item_encoder, dataset, item_index, ks=(10, 50, 100, 200, 500), device="cpu"):
    model.eval()
    user_ids = val_pos["user_id"].values
    biz_ids = val_pos["business_id"].values
    user_idx_arr = np.array([user_encoder.encode(u) for u in user_ids], dtype=np.int64)
    pos_item_idx_arr = np.array([item_encoder.encode(b) for b in biz_ids], dtype=np.int64)

    user_idx_t = torch.from_numpy(user_idx_arr).to(device)
    user_num_t = dataset.user_num_t[user_idx_t.cpu()].to(device)
    user_cuisine_t = dataset.user_cuisine_t[user_idx_t.cpu()].to(device)
    user_emb_all = []
    bs = 4096
    for s in range(0, len(user_idx_arr), bs):
        e = min(s + bs, len(user_idx_arr))
        u_emb = model.encode_user(user_idx_t[s:e], user_num_t[s:e], user_cuisine_t[s:e]).cpu().numpy()
        user_emb_all.append(u_emb)
    user_emb = np.vstack(user_emb_all)
    scores = user_emb @ item_index.T

    metrics = {}
    for k in ks:
        top_k_idx = np.argpartition(-scores, k, axis=1)[:, :k]
        hits = np.any(top_k_idx == pos_item_idx_arr[:, None], axis=1)
        metrics[f"Recall@{k}"] = float(hits.mean())
    return metrics

val_pos_filtered = val_df[val_df["stars"] >= 4].copy()
val_pos_filtered = val_pos_filtered[val_pos_filtered["user_id"].isin(user_encoder.id_to_idx)]
val_pos_filtered = val_pos_filtered[val_pos_filtered["business_id"].isin(item_encoder.id_to_idx)]
print(f"val positives: {len(val_pos_filtered)}")

metrics = eval_recall_at_k(model, val_pos_filtered, user_encoder, item_encoder, train_ds, item_index, device=DEVICE)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

with open(MODELS_DIR / "two_tower_v3_metrics.json", "w") as f:
    json.dump({"config": CONFIG, "history": history, "recall": metrics}, f, indent=2)

Compare v2 (BCE 1:4) vs v3 (in-batch softmax) side by side.

In [ ]:
with open(MODELS_DIR / "two_tower_metrics.json") as f:
    v2 = json.load(f)["recall"]
with open(MODELS_DIR / "two_tower_v3_metrics.json") as f:
    v3 = json.load(f)["recall"]

print(f"{'K':<6}{'v2 BCE 1:4':<14}{'v3 in-batch':<14}{'delta':<10}")
for k in ["Recall@10", "Recall@50", "Recall@100", "Recall@200", "Recall@500"]:
    a = v2[k]; b = v3[k]
    print(f"{k:<6}{a:<14.4f}{b:<14.4f}{b-a:+.4f}")